# EDA — Mi Spotify Wrapped (DWH Analysis)

**Proyecto**: dwh-spotify-wrapped  
**Materia**: Bases de Datos II — Universidad de Pamplona  
**Autor**: Didier  
**Fecha**: 2026-05-18

Análisis exploratorio del Data Warehouse dimensional construido sobre Cloud SQL (PostgreSQL 16).  
Todas las horas pico usan la columna `_cot` (Colombia, UTC-5) para reflejar el comportamiento real del oyente.

---
**Prerequisito**: subir `colab-eda-key.json` (Service Account `sa-colab-eda`) al runtime de Colab.

## 1. Setup — Dependencias y conexión

In [ ]:
!pip install "cloud-sql-python-connector[pg8000]" SQLAlchemy pandas matplotlib seaborn -q

In [ ]:
from google.colab import files
uploaded = files.upload()  # subir colab-eda-key.json

import os
key_file = list(uploaded.keys())[0]
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = f"/content/{key_file}"
print(f"Credenciales cargadas: {key_file}")

In [ ]:
from google.cloud.sql.connector import Connector
import sqlalchemy

# ── Configuración ──────────────────────────────────────────────────────────
INSTANCE_CONNECTION_NAME = "dwh-spotify-wrapped:us-central1:spotify-postgres"
DB_USER     = "postgres"
DB_PASSWORD = ""   # completar con la contraseña del usuario postgres
DB_NAME     = "postgres"
# ──────────────────────────────────────────────────────────────────────────

connector = Connector()

def getconn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASSWORD,
        db=DB_NAME,
        ip_type="PUBLIC",
    )

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)
print("Conexión establecida con Cloud SQL")

## 2. Carga de datos del DWH

In [ ]:
import pandas as pd

df_facts = pd.read_sql("""
    SELECT
        f.history_id,
        f.played_at,
        f.hour_of_day,
        f.day_of_week,
        f.hour_of_day_cot,
        f.day_of_week_cot,
        f.context_type,
        t.name       AS track_name,
        t.popularity AS track_popularity,
        t.duration_ms,
        t.explicit,
        a.name       AS artist_name,
        a.popularity AS artist_popularity,
        a.genres
    FROM dwh.fact_listening_history f
    JOIN dwh.dim_tracks  t ON f.track_id  = t.track_id
    JOIN dwh.dim_artists a ON t.artist_id = a.artist_id
    ORDER BY f.played_at DESC
""", engine)

df_artists = pd.read_sql("SELECT * FROM dwh.dim_artists", engine)
df_tracks  = pd.read_sql("SELECT * FROM dwh.dim_tracks",  engine)
df_audit   = pd.read_sql("SELECT * FROM dwh.etl_audit ORDER BY started_at", engine)

print(f"Facts         : {len(df_facts):>5} filas")
print(f"Artistas      : {len(df_artists):>5} filas")
print(f"Canciones     : {len(df_tracks):>5} filas")
print(f"Runs ETL      : {len(df_audit):>5} registros")
df_facts.head()

## 3. ETL Audit — Historial de ejecuciones

In [ ]:
audit_display = df_audit[[
    "started_at", "status", "history_new", "history_skipped",
    "artists_new", "tracks_new", "error_message"
]].copy()
audit_display["started_at"] = pd.to_datetime(audit_display["started_at"]).dt.strftime("%Y-%m-%d %H:%M")
print(f"Runs exitosos : {(df_audit.status == 'success').sum()}")
print(f"Total tracks ingresados: {df_audit.history_new.sum()}")
audit_display

## 4. Horas pico de escucha (COT — UTC-5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="viridis")

hourly = df_facts.groupby("hour_of_day_cot").size().reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly.index, hourly.values, color=sns.color_palette("viridis", 24))
ax.set_xlabel("Hora del día (COT — UTC-5)")
ax.set_ylabel("Reproducciones")
ax.set_title("Distribución de reproducciones por hora (Colombia)")
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

peak_hour = hourly.idxmax()
print(f"Hora pico: {peak_hour}:00 COT con {hourly[peak_hour]} reproducciones")

## 5. Días de la semana más activos (COT)

In [ ]:
DAY_ORDER = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
DAY_ES    = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

daily = (df_facts
         .groupby("day_of_week_cot")
         .size()
         .reindex(DAY_ORDER, fill_value=0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(DAY_ES, daily.values, color=sns.color_palette("magma", 7))
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Reproducciones")
ax.set_title("Reproducciones por día de la semana (COT)")
plt.tight_layout()
plt.show()

## 6. Top 10 artistas más escuchados

In [ ]:
top_artists = (df_facts
               .groupby("artist_name")
               .size()
               .sort_values(ascending=False)
               .head(10))

fig, ax = plt.subplots(figsize=(10, 5))
top_artists.sort_values().plot(kind="barh", ax=ax, color=sns.color_palette("rocket", 10))
ax.set_xlabel("Reproducciones")
ax.set_title("Top 10 artistas más escuchados")
plt.tight_layout()
plt.show()

## 7. Top 10 canciones más reproducidas

In [ ]:
top_tracks = (df_facts
              .groupby(["track_name", "artist_name"])
              .size()
              .sort_values(ascending=False)
              .head(10)
              .reset_index(name="plays"))

top_tracks["label"] = top_tracks["track_name"] + " — " + top_tracks["artist_name"]

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top_tracks["label"].iloc[::-1], top_tracks["plays"].iloc[::-1],
        color=sns.color_palette("mako", 10))
ax.set_xlabel("Reproducciones")
ax.set_title("Top 10 canciones más reproducidas")
plt.tight_layout()
plt.show()

## 8. Popularidad promedio de artistas escuchados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_facts["artist_popularity"].dropna().plot(
    kind="hist", bins=20, ax=axes[0],
    color="steelblue", edgecolor="white", title="Distribución popularidad artistas"
)
df_facts["track_popularity"].dropna().plot(
    kind="hist", bins=20, ax=axes[1],
    color="coral", edgecolor="white", title="Distribución popularidad canciones"
)
for ax in axes:
    ax.set_xlabel("Popularidad (0–100)")
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

print(f"Popularidad promedio artistas : {df_facts.artist_popularity.mean():.1f}")
print(f"Popularidad promedio canciones: {df_facts.track_popularity.mean():.1f}")

## 9. Contenido explícito vs no explícito

In [ ]:
explicit_counts = df_facts["explicit"].value_counts()
labels = ["Explícito" if v else "No explícito" for v in explicit_counts.index]

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(explicit_counts.values, labels=labels, autopct="%1.1f%%",
       colors=["#e74c3c", "#2ecc71"], startangle=90)
ax.set_title("Proporción de canciones explícitas")
plt.tight_layout()
plt.show()

## 10. Contexto de reproducción (playlist, álbum, radio)

In [ ]:
ctx = df_facts["context_type"].fillna("desconocido").value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ctx.plot(kind="bar", ax=ax, color=sns.color_palette("Set2", len(ctx)), edgecolor="white")
ax.set_xlabel("Contexto")
ax.set_ylabel("Reproducciones")
ax.set_title("Reproducciones por contexto")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

---
## Cierre de conexión

In [ ]:
connector.close()
print("Conexión cerrada.")